# Lab 02 · Mesmos dados. Três algoritmos
**Machine Learning na Prática · Dia 1 · Módulo 03: KNN, Árvores & Random Forest** · ~25 min

Os dados já chegam **prontos**: o clássico **Iris** (150 flores, 4 medidas, 3 espécies), sem faltantes nem sujeira.
O foco é comparar classificadores com honestidade.

| Etapa | O que fazer | Tempo |
|---|---|---|
| 1. Olhar e separar | explorar o Iris e separar treino/teste com estratificação | 5 min |
| 2. Comparar | KNN × Árvore × Random Forest com validação cruzada no treino | 8 min |
| 3. Ajustar | variar K e profundidade: qualidade e variância | 7 min |
| 4. Avaliar | o escolhido no teste: matriz de confusão e métricas | 5 min |

Procure as células marcadas com **`# TODO`** e substitua os `___`.

## 0 · Preparação (célula pronta)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, precision_score, recall_score

plt.rcParams['figure.figsize'] = (8, 4)

iris = load_iris(as_frame=True)
X, y = iris.data, iris.target
nomes = iris.target_names
X.head()

## 1 · Olhar e separar

In [ ]:
# TODO: quantas flores de cada espécie?
print(y.___())

plt.scatter(X['petal length (cm)'], X['petal width (cm)'], c=y, cmap='viridis', alpha=.8)
plt.xlabel('comprimento da pétala'); plt.ylabel('largura da pétala'); plt.show()

**Estratificar** mantém a mesma proporção de espécies no treino e no teste — importante quando há poucas amostras.

In [ ]:
# TODO: 25% para teste, estratificando pelo alvo
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=___)
print(y_tr.value_counts().to_dict(), y_te.value_counts().to_dict())

## 2 · KNN × Árvore × Random Forest
O KNN usa **distâncias**, então vai num `Pipeline` com padronização. Árvores não ligam para escala.
Comparamos pela **acurácia média da validação cruzada no treino**.

In [ ]:
modelos = {
    # TODO: KNN com padronização dentro do pipeline
    'KNN (K=5)': Pipeline([('padronizar', ___), ('knn', KNeighborsClassifier(n_neighbors=5))]),
    'Árvore': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
}

for nome, modelo in modelos.items():
    # TODO: acurácia da validação cruzada com 5 folds, só no treino
    acc = cross_val_score(modelo, ___, ___, cv=___, scoring='accuracy')
    print(f'{nome:<14} {acc.mean():.3f} ± {acc.std():.3f}')

## 3 · Variar K e profundidade
Compare a acurácia **no treino** com a da **validação cruzada**. Quando as duas se afastam, o modelo está decorando.

In [ ]:
def treino_vs_cv(modelo):
    acc_tr = modelo.fit(X_tr, y_tr).score(X_tr, y_tr)
    acc_cv = cross_val_score(modelo, X_tr, y_tr, cv=5).mean()
    return acc_tr, acc_cv

ks = [1, 3, 5, 9, 15, 31, 61]
# TODO: monte o KNN com K = k
res_k = [treino_vs_cv(Pipeline([('padronizar', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=___))])) for k in ks]

profs = [1, 2, 3, 4, 6, None]
# TODO: monte a árvore com profundidade máxima = p
res_p = [treino_vs_cv(DecisionTreeClassifier(max_depth=___, random_state=42)) for p in profs]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ks, [r[0] for r in res_k], 'o-', label='treino'); ax[0].plot(ks, [r[1] for r in res_k], 'o-', label='validação cruzada')
ax[0].set_xscale('log'); ax[0].set_xlabel('K'); ax[0].set_title('KNN'); ax[0].legend()
rot = [str(p) for p in profs]
ax[1].plot(rot, [r[0] for r in res_p], 'o-', label='treino'); ax[1].plot(rot, [r[1] for r in res_p], 'o-', label='validação cruzada')
ax[1].set_xlabel('profundidade máxima'); ax[1].set_title('Árvore'); ax[1].legend()
plt.show()

**Para pensar:** com K = 1 e com a árvore sem limite (`None`), a acurácia no treino é 100%. Isso significa que são os melhores modelos?

In [ ]:
arvore = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_tr, y_tr)
plt.figure(figsize=(10, 5))
plot_tree(arvore, feature_names=X.columns, class_names=nomes, filled=True); plt.show()

## 4 · Avaliar o escolhido no teste
Escolha um modelo **pela validação cruzada** (não pelo teste!). Só agora o teste é usado — uma única vez.

In [ ]:
# TODO: escolha o modelo (ex.: modelos['Random Forest']) e treine no treino inteiro
escolhido = ___.fit(X_tr, y_tr)
pred = escolhido.predict(X_te)

ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=nomes); plt.show()
# TODO: precision, recall e F1 por classe
print(___(y_te, pred, target_names=nomes))

## 5 · Desafios extras (se sobrar tempo)
### 5.1 Precision × recall: quem escolhe é o limiar
No Iris tudo é fácil. Num problema médico, errar um caso positivo custa caro.
Use o dataset de **câncer de mama** do Scikit-learn (positivo = tumor maligno) e veja o efeito do limiar.

In [ ]:
bc = load_breast_cancer()
Xb, yb = bc.data, (bc.target == 0).astype(int)          # 1 = maligno
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.25, random_state=42, stratify=yb)
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xb_tr, yb_tr)
proba = rf.predict_proba(Xb_te)[:, 1]                    # probabilidade de ser maligno

for limiar in [0.2, 0.35, 0.5, 0.65, 0.8]:
    # TODO: positivo quando a probabilidade passa do limiar
    prev = (proba >= ___).astype(int)
    print(f'limiar {limiar:.2f} → precision {precision_score(yb_te, prev):.2f} | recall {recall_score(yb_te, prev):.2f}')

**Para pensar:** numa triagem de câncer, você prefere limiar alto ou baixo? E num filtro de spam?

### 5.2 “Perto” depende de como você mede
O KNN usa a distância euclidiana por padrão. Compare com a de Manhattan.

In [ ]:
for metrica in ['euclidean', 'manhattan']:
    # TODO: passe a métrica para o KNN
    knn = Pipeline([('padronizar', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5, metric=___))])
    print(metrica, cross_val_score(knn, X_tr, y_tr, cv=5).mean().round(3))

## Para discutir
1. O modelo que foi melhor **no treino** deve necessariamente ser o escolhido?
2. Por que escolher pela validação cruzada e só depois olhar o teste?
3. Onde o modelo errou na matriz de confusão? Essas espécies se parecem no gráfico de pétalas?